# Fraud Detection — CRISP-DM Pipeline
### IS 455 | Chapters 1–17

---

## 1. Business Understanding

**Problem:** Identify fraudulent orders before fulfillment to reduce chargebacks
and protect revenue.

**Target variable:** `orders.is_fraud` (binary: 1 = fraud, 0 = legitimate)

**Success metric:** ROC-AUC ≥ 0.90 on held-out test set. Precision on the fraud
class matters more than recall — a false positive blocks a legitimate order,
which has a real cost.

**Deployment plan:**
The trained model artifact (`jobs/fraud_model.pkl`) is loaded by
`jobs/run_inference.py`, which scores new orders and writes predictions into
`order_predictions` in `shop.db`. The web app surfaces the top-risk orders
in the Priority Queue.


## 2. Imports & Setup

In [ ]:
import sqlite3
import warnings
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.compose import ColumnTransformer
from sklearn.ensemble import GradientBoostingClassifier, RandomForestClassifier
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    ConfusionMatrixDisplay,
    classification_report,
    roc_auc_score,
    roc_curve,
)
from sklearn.model_selection import GridSearchCV, StratifiedKFold, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.tree import DecisionTreeClassifier

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid", palette="muted")

DB_PATH   = Path("..") / "shop.db"
MODEL_DIR = Path("..") / "jobs"
MODEL_PATH = MODEL_DIR / "fraud_model.pkl"

print("DB exists:", DB_PATH.exists())
print("Model dir exists:", MODEL_DIR.exists())


## 3. Data Understanding (Ch. 6, 8)

Load the operational tables from `shop.db` and explore their structure,
distributions, and relationships.


In [ ]:
conn = sqlite3.connect(DB_PATH)

orders = pd.read_sql("SELECT * FROM orders", conn)
customers = pd.read_sql(
    "SELECT customer_id, customer_segment, loyalty_tier, is_active, created_at FROM customers",
    conn,
)
items_agg = pd.read_sql(
    """
    SELECT
        order_id,
        COUNT(*)                   AS item_count,
        COUNT(DISTINCT product_id) AS unique_products,
        SUM(line_total)            AS items_total,
        MAX(unit_price)            AS max_unit_price,
        AVG(unit_price)            AS avg_unit_price
    FROM order_items
    GROUP BY order_id
    """,
    conn,
)

conn.close()

print("orders:   ", orders.shape)
print("customers:", customers.shape)
print("items_agg:", items_agg.shape)
orders.head(3)


In [ ]:
# Schema / dtypes
orders.info()


In [ ]:
# Missing values
missing = orders.isnull().sum()
missing[missing > 0]


In [ ]:
# Target distribution
fraud_counts = orders["is_fraud"].value_counts()
fraud_rate   = orders["is_fraud"].mean()

print(f"Fraud rate: {fraud_rate:.2%}  ({fraud_counts[1]:,} fraud / {fraud_counts[0]:,} legit)")

fig, ax = plt.subplots(figsize=(4, 3))
fraud_counts.plot(kind="bar", color=["steelblue", "salmon"], ax=ax)
ax.set_xticklabels(["Legit (0)", "Fraud (1)"], rotation=0)
ax.set_title("Class Distribution — is_fraud")
ax.set_ylabel("Count")
plt.tight_layout()
plt.show()


In [ ]:
# Numeric feature distributions by class (Ch. 8 EDA)
num_cols = ["order_subtotal", "shipping_fee", "order_total", "risk_score"]
fig, axes = plt.subplots(1, len(num_cols), figsize=(14, 3))
for ax, col in zip(axes, num_cols):
    orders.groupby("is_fraud")[col].plot(kind="kde", ax=ax, legend=True)
    ax.set_title(col)
    ax.set_xlabel("")
plt.suptitle("Feature Distributions by Fraud Label", y=1.02)
plt.tight_layout()
plt.show()


In [ ]:
# Categorical breakdown
cat_cols = ["payment_method", "device_type", "ip_country"]
fig, axes = plt.subplots(1, len(cat_cols), figsize=(14, 4))
for ax, col in zip(axes, cat_cols):
    fraud_by_cat = (
        orders.groupby(col)["is_fraud"].mean().sort_values(ascending=False).head(10)
    )
    fraud_by_cat.plot(kind="bar", ax=ax, color="salmon")
    ax.set_title(f"Fraud Rate by {col}")
    ax.set_ylabel("Fraud Rate")
    ax.set_xlabel("")
    ax.tick_params(axis="x", rotation=45)
plt.tight_layout()
plt.show()


## 4. Data Preparation & Feature Engineering (Ch. 2–4, 7)

Join tables, engineer derived features, encode categoricals, scale numerics,
and split into train / test sets.


In [ ]:
# ── Join tables ──────────────────────────────────────────────────────────────
df = (
    orders
    .merge(customers, on="customer_id", how="left")
    .merge(items_agg,  on="order_id",   how="left")
)

# ── Derived features ─────────────────────────────────────────────────────────
df["billing_shipping_match"] = (df["billing_zip"] == df["shipping_zip"]).astype(int)
df["domestic_ip"]            = (df["ip_country"] == "US").astype(int)
df["promo_used"]             = df["promo_used"].fillna(0).astype(int)

df["order_dt"]   = pd.to_datetime(df["order_datetime"])
df["created_dt"] = pd.to_datetime(df["created_at"])
df["account_age_days"] = (df["order_dt"] - df["created_dt"]).dt.days.clip(lower=0)
df["order_hour"]       = df["order_dt"].dt.hour
df["is_weekend"]       = df["order_dt"].dt.dayofweek.isin([5, 6]).astype(int)

# ── Fill missing from left joins ──────────────────────────────────────────────
for col in ["item_count", "unique_products", "items_total", "max_unit_price", "avg_unit_price"]:
    df[col] = df[col].fillna(0)

df["account_age_days"] = df["account_age_days"].fillna(df["account_age_days"].median())

print(f"Dataset shape after joining: {df.shape}")
df[["order_id", "is_fraud", "billing_shipping_match", "account_age_days", "order_hour"]].head()


In [ ]:
# ── Feature lists ─────────────────────────────────────────────────────────────
NUMERIC_FEATURES = [
    "order_subtotal", "shipping_fee", "order_total", "risk_score",
    "item_count", "unique_products", "max_unit_price", "avg_unit_price",
    "billing_shipping_match", "domestic_ip", "promo_used",
    "account_age_days", "order_hour", "is_weekend",
]
CATEGORICAL_FEATURES = [
    "payment_method", "device_type", "customer_segment", "loyalty_tier",
]
TARGET = "is_fraud"

X = df[NUMERIC_FEATURES + CATEGORICAL_FEATURES].copy()
y = df[TARGET].copy()

print(f"X shape: {X.shape}  |  Fraud rate: {y.mean():.2%}")


In [ ]:
# ── Train / test split (stratified to preserve class balance) ─────────────────
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
print(f"Train: {X_train.shape}  fraud={y_train.mean():.2%}")
print(f"Test:  {X_test.shape}   fraud={y_test.mean():.2%}")


In [ ]:
# ── Preprocessing pipeline ────────────────────────────────────────────────────
preprocessor = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), NUMERIC_FEATURES),
        ("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=False), CATEGORICAL_FEATURES),
    ]
)


## 5. Classification Modeling (Ch. 13)

Baseline models: Logistic Regression and Decision Tree.
`class_weight='balanced'` corrects for the class imbalance we observed above.


In [ ]:
def evaluate(name, pipeline, X_tr, y_tr, X_te, y_te, ax_roc=None):
    """Fit, predict, print report, optionally plot ROC curve."""
    pipeline.fit(X_tr, y_tr)
    y_pred = pipeline.predict(X_te)
    y_prob = pipeline.predict_proba(X_te)[:, 1]
    auc    = roc_auc_score(y_te, y_prob)

    print(f"\n{'─'*55}")
    print(f"  {name}  |  ROC-AUC: {auc:.4f}")
    print(f"{'─'*55}")
    print(classification_report(y_te, y_pred, target_names=["Legit", "Fraud"]))

    fig, axes = plt.subplots(1, 2, figsize=(9, 3))
    ConfusionMatrixDisplay.from_predictions(
        y_te, y_pred, display_labels=["Legit", "Fraud"], ax=axes[0]
    )
    axes[0].set_title(f"Confusion Matrix — {name}")

    fpr, tpr, _ = roc_curve(y_te, y_prob)
    axes[1].plot(fpr, tpr, label=f"AUC = {auc:.3f}")
    axes[1].plot([0, 1], [0, 1], "k--")
    axes[1].set_xlabel("FPR"); axes[1].set_ylabel("TPR")
    axes[1].set_title(f"ROC Curve — {name}")
    axes[1].legend()
    plt.tight_layout()
    plt.show()

    return auc, pipeline


In [ ]:
results = {}

lr_pipe = Pipeline([
    ("pre", preprocessor),
    ("clf", LogisticRegression(class_weight="balanced", max_iter=1000, random_state=42)),
])
results["Logistic Regression"] = evaluate(
    "Logistic Regression", lr_pipe, X_train, y_train, X_test, y_test
)


In [ ]:
dt_pipe = Pipeline([
    ("pre", preprocessor),
    ("clf", DecisionTreeClassifier(class_weight="balanced", max_depth=8, random_state=42)),
])
results["Decision Tree"] = evaluate(
    "Decision Tree", dt_pipe, X_train, y_train, X_test, y_test
)


## 6. Ensemble Methods (Ch. 14)

Random Forest (bagging) and Gradient Boosting (boosting) typically outperform
single-tree models by reducing variance and bias respectively.


In [ ]:
rf_pipe = Pipeline([
    ("pre", preprocessor),
    ("clf", RandomForestClassifier(
        n_estimators=100, class_weight="balanced", random_state=42, n_jobs=-1
    )),
])
results["Random Forest"] = evaluate(
    "Random Forest", rf_pipe, X_train, y_train, X_test, y_test
)


In [ ]:
gb_pipe = Pipeline([
    ("pre", preprocessor),
    ("clf", GradientBoostingClassifier(
        n_estimators=100, learning_rate=0.1, max_depth=4, random_state=42
    )),
])
results["Gradient Boosting"] = evaluate(
    "Gradient Boosting", gb_pipe, X_train, y_train, X_test, y_test
)


## 7. Model Evaluation, Selection & Tuning (Ch. 15)

Compare all models by ROC-AUC, then tune the best one with `GridSearchCV`
using 5-fold stratified cross-validation.


In [ ]:
# Summary table
summary = pd.DataFrame(
    [(name, auc) for name, (auc, _) in results.items()],
    columns=["Model", "ROC-AUC"],
).sort_values("ROC-AUC", ascending=False).reset_index(drop=True)

print(summary.to_string(index=False))

fig, ax = plt.subplots(figsize=(6, 3))
summary.plot(kind="barh", x="Model", y="ROC-AUC", ax=ax, color="steelblue", legend=False)
ax.set_xlim(0.5, 1.0)
ax.set_title("Model Comparison — ROC-AUC")
ax.axvline(0.9, color="red", linestyle="--", label="Target (0.90)")
ax.legend()
plt.tight_layout()
plt.show()

best_name = summary.iloc[0]["Model"]
print(f"\nBest model: {best_name}")


In [ ]:
# ── GridSearchCV on Random Forest ────────────────────────────────────────────
param_grid = {
    "clf__n_estimators":    [100, 200],
    "clf__max_depth":       [None, 15, 30],
    "clf__min_samples_leaf":[1, 5],
}

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

grid = GridSearchCV(
    Pipeline([
        ("pre", preprocessor),
        ("clf", RandomForestClassifier(class_weight="balanced", random_state=42, n_jobs=-1)),
    ]),
    param_grid,
    cv=cv,
    scoring="roc_auc",
    n_jobs=-1,
    verbose=1,
)
grid.fit(X_train, y_train)

print(f"Best params: {grid.best_params_}")
print(f"CV ROC-AUC:  {grid.best_score_:.4f}")

best_model = grid.best_estimator_
tuned_auc, _ = evaluate(
    "Tuned Random Forest", best_model, X_train, y_train, X_test, y_test
)
results["Tuned Random Forest"] = (tuned_auc, best_model)


## 8. Feature Selection (Ch. 16)

Two complementary approaches:
1. **SelectKBest** (univariate F-test) — statistical filter
2. **Random Forest feature importances** — model-based ranking


In [ ]:
# Fit preprocessor on training data to get transformed feature names
pre_fit = preprocessor.fit(X_train, y_train)
cat_names = pre_fit.named_transformers_["cat"].get_feature_names_out(CATEGORICAL_FEATURES)
all_feature_names = np.array(NUMERIC_FEATURES + list(cat_names))

X_train_pre = pre_fit.transform(X_train)
X_test_pre  = pre_fit.transform(X_test)

# SelectKBest
k = 10
selector = SelectKBest(f_classif, k=k)
selector.fit(X_train_pre, y_train)
top_k_idx   = selector.get_support(indices=True)
top_k_names = all_feature_names[top_k_idx]
top_k_scores = selector.scores_[top_k_idx]

print(f"Top {k} features by F-score:")
for name, score in sorted(zip(top_k_names, top_k_scores), key=lambda x: -x[1]):
    print(f"  {name:<35} {score:.2f}")


In [ ]:
# Random Forest feature importances
rf_clf = best_model.named_steps["clf"]
importances = pd.Series(rf_clf.feature_importances_, index=all_feature_names)
top15 = importances.sort_values(ascending=False).head(15)

fig, ax = plt.subplots(figsize=(8, 5))
top15.sort_values().plot(kind="barh", ax=ax, color="steelblue")
ax.set_title("Top 15 Feature Importances — Tuned Random Forest")
ax.set_xlabel("Mean Decrease in Impurity")
plt.tight_layout()
plt.show()


## 9. Model Serialization (Ch. 17)

Save the best model pipeline (preprocessor + classifier) to disk with `joblib`.
`run_inference.py` loads this artifact to score new orders at runtime.


In [ ]:
joblib.dump(best_model, MODEL_PATH)
print(f"Model saved → {MODEL_PATH.resolve()}")
print(f"File size:    {MODEL_PATH.stat().st_size / 1024:.1f} KB")


In [ ]:
# Verify round-trip
loaded_model = joblib.load(MODEL_PATH)
sample = X_test.iloc[:5]
probs  = loaded_model.predict_proba(sample)[:, 1]
preds  = loaded_model.predict(sample)

pd.DataFrame({
    "order_id":       df.loc[sample.index, "order_id"].values,
    "fraud_prob":     probs.round(4),
    "predicted_fraud": preds,
    "actual_fraud":   y_test.iloc[:5].values,
})


## 10. Deployment Integration

The model is consumed by `jobs/run_inference.py`, which:
1. Loads `fraud_model.pkl`
2. Queries `shop.db` for unscored orders
3. Builds the same feature matrix used here
4. Writes `fraud_probability` and `predicted_fraud` into `order_predictions`
5. The web app reads these and displays them in the Priority Queue

**To score new orders after training, click "Run Scoring" in the web app**
or run directly:

```bash
python jobs/run_inference.py
```


In [ ]:
# Final model summary
print("=" * 55)
print("FINAL MODEL SUMMARY")
print("=" * 55)
print(f"  Algorithm : Tuned Random Forest")
print(f"  Best params: {grid.best_params_}")
print(f"  Test ROC-AUC: {tuned_auc:.4f}")
print(f"  Artifact  : {MODEL_PATH.resolve()}")
print(f"  Features  : {len(NUMERIC_FEATURES + CATEGORICAL_FEATURES)} input → {X_train_pre.shape[1]} encoded")
print("=" * 55)
